# 🏨 StayWise AI

## AI-Powered Hotel Customer Service & Booking Assistant

StayWise AI is a Generative AI prototype designed to demonstrate how AI can automate hotel customer service and support basic booking workflows.

The V1 prototype can answer hotel-related questions, provide room and facility information, identify booking requests, extract room type and number of nights, and calculate booking prices using Python-based business logic.

### 🚀 V1 Features

- 🤖 AI-powered hotel customer service
- 💬 Hotel FAQ and facility assistance
- 🛏️ Room information and recommendations
- 📅 Booking-intent detection
- 🔎 Room type and number-of-nights extraction
- 💰 Automatic booking price calculation
- 🧠 AI language understanding + Python business logic
- 📊 Basic customer booking workflow

### 🛠️ Technology

- Python
- Google Colab
- Hugging Face Transformers
- Qwen2.5-0.5B-Instruct
- Gradio
- Pandas

### 🔄 Project Roadmap

- **V1 — AI Hotel Assistant:** Customer service, hotel FAQs, booking intent and price calculation
- **V2 — RAG Knowledge Base:** Connect the AI to hotel documents, policies, FAQs and website content
- **V3 — Interactive Chatbot UI:** Add conversation memory and multi-step customer interactions
- **V4 — Website Deployment:** Deploy the assistant as a customer-facing web chatbot
- **V5 — Booking API Integration:** Connect the assistant with a real booking system/API
- **V6 — Analytics Dashboard:** Track customer questions, booking intent, conversations and business insights
- **V7 — Multi-Business SaaS Architecture:** Extend the platform to hotels, restaurants, e-commerce and other businesses

### 💡 Business Concept

The long-term goal is to transform StayWise AI from a hotel chatbot into a reusable AI customer-service platform that can understand customer requests, connect with business data, automate workflows and generate actionable business insights.

> ⚠️ **Disclaimer:** V1 is a portfolio prototype using fictional hotel data. It is not connected to a real hotel or booking system.

In [9]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu \
pypdf beautifulsoup4 requests gradio pandas matplotlib plotly flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 20.7 MB/s eta 0:00:00


In [10]:
import torch

print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("✅ GPU detected")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU not detected")
    print("The project can still run, but AI responses may be slower.")

PyTorch: 2.11.0+cpu
⚠️ GPU not detected
The project can still run, but AI responses may be slower.


In [11]:
from transformers import pipeline

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

print("✅ AI model loaded")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ AI model loaded


In [12]:
messages = [
    {
        "role": "system",
        "content": "You are a friendly hotel receptionist."
    },
    {
        "role": "user",
        "content": "Welcome me to a hotel in one sentence."
    }
]

result = generator(
    messages,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.4
)

print(result[0]["generated_text"][-1]["content"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Welcome you to our luxurious and welcoming hotel, where comfort and convenience await with every room and service.


In [13]:
hotel = {
    "name": "Grand Berlin Hotel",
    "location": "Berlin, Germany",
    "check_in": "3:00 PM",
    "check_out": "11:00 AM",
    "breakfast": "6:30 AM - 10:30 AM",
    "parking": "€20 per night",
    "wifi": "Free Wi-Fi",
    "pets": "Pets are allowed for €15 per night",
    "gym": "Open 24 hours",
    "pool": "7:00 AM - 10:00 PM",
    "cancellation": "Free cancellation up to 48 hours before check-in",
    "airport_transfer": "Airport transfer is available on request."
}

print("✅ Hotel information created")

✅ Hotel information created


In [14]:
rooms = {
    "Standard Room": {
        "price": 140,
        "max_guests": 2,
        "description": "Comfortable room with queen bed and city view."
    },

    "Deluxe Room": {
        "price": 190,
        "max_guests": 3,
        "description": "Spacious room with king bed and city view."
    },

    "Family Suite": {
        "price": 260,
        "max_guests": 4,
        "description": "Large suite suitable for families."
    },

    "Executive Suite": {
        "price": 350,
        "max_guests": 4,
        "description": "Premium suite with living area and executive amenities."
    }
}

print("✅ Room information created")

✅ Room information created


In [15]:
def build_hotel_knowledge(hotel, rooms):

    text = f"""
HOTEL NAME:
{hotel['name']}

LOCATION:
{hotel['location']}

CHECK-IN:
{hotel['check_in']}

CHECK-OUT:
{hotel['check_out']}

BREAKFAST:
{hotel['breakfast']}

PARKING:
{hotel['parking']}

WIFI:
{hotel['wifi']}

PETS:
{hotel['pets']}

GYM:
{hotel['gym']}

POOL:
{hotel['pool']}

CANCELLATION:
{hotel['cancellation']}

AIRPORT TRANSFER:
{hotel['airport_transfer']}

ROOMS:
"""

    for room, details in rooms.items():

        text += f"""
{room}
Price: €{details['price']} per night
Maximum guests: {details['max_guests']}
Description: {details['description']}
"""

    return text


hotel_knowledge = build_hotel_knowledge(
    hotel,
    rooms
)

print("✅ Hotel knowledge created")
print(hotel_knowledge)

✅ Hotel knowledge created

HOTEL NAME:
Grand Berlin Hotel

LOCATION:
Berlin, Germany

CHECK-IN:
3:00 PM

CHECK-OUT:
11:00 AM

BREAKFAST:
6:30 AM - 10:30 AM

PARKING:
€20 per night

WIFI:
Free Wi-Fi

PETS:
Pets are allowed for €15 per night

GYM:
Open 24 hours

POOL:
7:00 AM - 10:00 PM

CANCELLATION:
Free cancellation up to 48 hours before check-in

AIRPORT TRANSFER:
Airport transfer is available on request.

ROOMS:

Standard Room
Price: €140 per night
Maximum guests: 2
Description: Comfortable room with queen bed and city view.

Deluxe Room
Price: €190 per night
Maximum guests: 3
Description: Spacious room with king bed and city view.

Family Suite
Price: €260 per night
Maximum guests: 4
Description: Large suite suitable for families.

Executive Suite
Price: €350 per night
Maximum guests: 4
Description: Premium suite with living area and executive amenities.



In [16]:
SYSTEM_PROMPT = """
You are StayWise AI, a professional virtual hotel receptionist.

Your responsibilities:

1. Answer hotel questions.
2. Explain hotel facilities.
3. Explain room options.
4. Recommend rooms.
5. Help with booking enquiries.
6. Be polite and professional.

IMPORTANT:

- Only use the information provided in the hotel knowledge.
- Never invent hotel prices.
- Never invent facilities.
- Never claim that a booking is confirmed.
- If information is unavailable, say that you do not have that information.
- Keep responses concise.
"""

print("✅ AI instructions created")

✅ AI instructions created


In [17]:
def generate_ai_response(
    user_message,
    context=None
):

    if context is None:
        context = hotel_knowledge

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
            + "\n\nHOTEL KNOWLEDGE:\n"
            + context
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    result = generator(
        messages,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3
    )

    generated = result[0]["generated_text"]

    if isinstance(generated, list):
        return generated[-1]["content"].strip()

    return str(generated).strip()

In [18]:
questions = [
    "What time is check-in?",
    "Do you have parking?",
    "Are pets allowed?",
    "What rooms can accommodate four people?"
]

for question in questions:

    print("CUSTOMER:", question)

    answer = generate_ai_response(question)

    print("STAYWISE AI:", answer)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CUSTOMER: What time is check-in?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: The check-in time at Grand Berlin Hotel is 3:00 PM.
------------------------------------------------------------
CUSTOMER: Do you have parking?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: Yes, we offer free parking at Grand Berlin Hotel. You can park your vehicle in the parking garage or nearby areas. Please note that parking fees apply.
------------------------------------------------------------
CUSTOMER: Are pets allowed?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: Yes, pets are allowed for an additional fee of €15 per night.
------------------------------------------------------------
CUSTOMER: What rooms can accommodate four people?
STAYWISE AI: The Executive Suite offers spacious accommodation for four people. It includes a living area, executive amenities, and a large city view. The price for an Executive Suite is €350 per night.
------------------------------------------------------------


In [19]:
import re

def detect_booking_request(message):

    message_lower = message.lower()

    booking_words = [
        "book",
        "booking",
        "reserve",
        "reservation"
    ]

    is_booking = any(
        word in message_lower
        for word in booking_words
    )

    if not is_booking:
        return None

    nights_match = re.search(
        r'(\d+)\s*(?:night|nights)',
        message_lower
    )

    nights = None

    if nights_match:
        nights = int(nights_match.group(1))

    room_type = None

    for room in rooms.keys():

        if room.lower() in message_lower:
            room_type = room
            break

    if room_type is None:

        if "standard" in message_lower:
            room_type = "Standard Room"

        elif "deluxe" in message_lower:
            room_type = "Deluxe Room"

        elif "family" in message_lower:
            room_type = "Family Suite"

        elif "executive" in message_lower:
            room_type = "Executive Suite"

    return {
        "is_booking": True,
        "room_type": room_type,
        "nights": nights
    }

print("✅ Booking detector ready")

✅ Booking detector ready


In [20]:
def handle_booking_request(message):

    booking_request = detect_booking_request(message)

    if booking_request is None:
        return None

    room_type = booking_request["room_type"]
    nights = booking_request["nights"]

    if room_type is None:

        return """
🏨 I'd be happy to help you book a room.

Available rooms:

• Standard Room
• Deluxe Room
• Family Suite
• Executive Suite

Which room would you like?
"""

    if nights is None:

        return f"""
Sure! I can help you book the {room_type}.

How many nights would you like to stay?
"""

    price_per_night = rooms[room_type]["price"]

    total_price = price_per_night * nights

    return f"""
🏨 Booking Request

Room: {room_type}
Nights: {nights}
Price per night: €{price_per_night}
Total price: €{total_price}

To continue, please provide:

1. Check-in date
2. Guest name
3. Number of guests
4. Email address

⚠️ This is a booking request. The reservation is not confirmed yet.
"""

In [21]:
def staywise_bot(message):

    booking_response = handle_booking_request(message)

    if booking_response is not None:
        return booking_response

    return generate_ai_response(message)

V1 completed

Customer
   ↓
AI
   ↓
Hotel information
   ↓
Answer

In [ ]:
# CELL 12 — Chat with StayWise AI

while True:
    user_input = input("YOU: ")

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("STAYWISE AI: Thank you for chatting with us. Goodbye!")
        break

    answer = generate_ai_response(user_input)
    print("STAYWISE AI:", answer)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: Hello! How can I assist you today?
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: I'm sorry, but I don't have specific information about the types of rooms available at Grand Berlin Hotel. However, based on the general description, you can expect to find various types of rooms such as standard rooms, deluxe rooms, family suites, and executive suites. Each type has its own features and accommodations, so it's best to contact the hotel directly or check their website for more detailed information.
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

STAYWISE AI: Sure! Here's how I can assist you:

**Check-In:** 3:00 PM
**Check-Out:** 11:00 AM
**Breakfast:** 6:30 AM - 10:30 AM
**Parking:** €20 per night
**WiFi:** Free Wi-Fi
**Pets:** Pets are allowed for €15 per night
**Gym:** Open 24 hours
**Pool:** 7:00 AM - 10:00 PM
**Cancellation:** Free cancellation up to 48 hours before check-in

I will help you book your stay at Grand Berlin Hotel. Please let me know if there are any specific dates or times you'd
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: Sure! Here's how we can assist you:

**Room Type:** Standard Room

**Price:** €140 per night

**Check-In Time:** 3:00 PM

**Check-Out Time:** 11:00 AM

**Breakfast:** 6:30 AM - 10:30 AM

**Parking:** Free parking available at Grand Berlin Hotel

**WiFi:** Free WiFi access available throughout the hotel

**Pets Allowed:** Pets are welcome for an additional fee of €15 per night

**Gym:** Open 24 hours

**Pool:** Available from 7:00 AM to 10:00 PM

Please let me know if there's anything else
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAYWISE AI: Sure! Here's how I can assist you:

**Check-In:** September 3rd at 6:30 AM

**Check-Out:** September 7th at 11:00 AM

**Breakfast:** 6:30 AM - 10:30 AM

**Parking:** Free parking available

**WiFi:** Free WiFi access

**Pets:** Pets are welcome for an additional fee of €15 per night

**Gym:** Open 24 hours

**Pool:** 7:00 AM - 10:00 PM

**Cancellation Policy:** Free cancellation up to 48 hours before check-in

**Airport Transfer:** Available upon request

I hope this helps
------------------------------------------------------------
